# Sieci MLP w PyTorch: klasyfikacja obrazów PlantVillage

Notebook realizuje doświadczenia dla klasyfikacji obrazów liści z bazy
[PlantVillage](https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset).

Zakres:

- klasyfikacja z co najmniej 5 klasami,
- MLP w PyTorch z jedną oraz dwiema warstwami ukrytymi,
- porównanie hiperparametrów: batch size, optimizer, learning rate, regularyzacja, augmentacja, aktywacje,
- standardowe wizualizacje: przykładowe obrazy, rozkład klas, przebiegi uczenia, confusion matrix,
- tabela końcowa z hiperparametrami i dokładnością,
- dodatkowy mały CNN do wizualizacji filtrów splotowych, ponieważ czysty MLP nie posiada filtrów konwolucyjnych.

## 1. Konfiguracja

Ustaw `DATA_DIR` na katalog zawierający klasy w podfolderach. Dla Kaggle często jest to jeden z wariantów:

- `plantvillage dataset/color`
- `plantvillage-dataset/plantvillage dataset/color`
- `PlantVillage`

Notebook wybiera pierwsze 5 klas alfabetycznie, ale listę `SELECTED_CLASSES` można wpisać ręcznie.

In [ ]:
from pathlib import Path
import random
import time
from copy import deepcopy

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image

import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

DATA_DIR = Path("plantvillage-dataset/plantvillage dataset/color")
IMG_SIZE = 64
NUM_CLASSES = 5
SELECTED_CLASSES = None  # np. ["Apple___Apple_scab", "Apple___Black_rot", ...]

In [ ]:
def find_image_root(preferred: Path) -> Path:
    candidates = [
        preferred,
        Path("plantvillage dataset/color"),
        Path("plantvillage-dataset/color"),
        Path("PlantVillage"),
        Path("data/plantvillage dataset/color"),
        Path("data/PlantVillage"),
    ]
    for path in candidates:
        if path.exists() and any(p.is_dir() for p in path.iterdir()):
            return path
    raise FileNotFoundError(
        "Nie znaleziono katalogu z danymi. Ustaw DATA_DIR na katalog z podfolderami klas."
    )

DATA_ROOT = find_image_root(DATA_DIR)
print("DATA_ROOT:", DATA_ROOT.resolve())

## 2. Wczytanie danych i wybór 5 klas

In [ ]:
base_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

full_dataset = datasets.ImageFolder(DATA_ROOT, transform=base_transform)
all_classes = full_dataset.classes

if SELECTED_CLASSES is None:
    selected_classes = all_classes[:NUM_CLASSES]
else:
    selected_classes = SELECTED_CLASSES

selected_class_ids = [full_dataset.class_to_idx[name] for name in selected_classes]
idx_to_new = {old_idx: new_idx for new_idx, old_idx in enumerate(selected_class_ids)}
selected_indices = [
    i for i, (_, y) in enumerate(full_dataset.samples)
    if y in selected_class_ids
]

print("Liczba klas w bazie:", len(all_classes))
print("Wybrane klasy:", selected_classes)
print("Liczba obrazów dla wybranych klas:", len(selected_indices))

In [ ]:
targets_old = np.array([full_dataset.samples[i][1] for i in selected_indices])
targets_new = np.array([idx_to_new[y] for y in targets_old])

train_idx, temp_idx, y_train, y_temp = train_test_split(
    selected_indices,
    targets_new,
    test_size=0.30,
    random_state=SEED,
    stratify=targets_new,
)
val_idx, test_idx, y_val, y_test = train_test_split(
    temp_idx,
    y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp,
)

print(len(train_idx), "train |", len(val_idx), "val |", len(test_idx), "test")

In [ ]:
class RemappedImageFolder(datasets.ImageFolder):
    def __init__(self, root, selected_old_ids, idx_to_new, transform=None):
        super().__init__(root, transform=transform)
        self.selected_old_ids = set(selected_old_ids)
        self.idx_to_new = idx_to_new

    def __getitem__(self, index):
        image, old_target = super().__getitem__(index)
        return image, self.idx_to_new[old_target]

def make_datasets(augment=False):
    train_tfms = [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
    ]
    if augment:
        train_tfms.extend([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(20),
            transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10),
            transforms.RandomAffine(degrees=0, translate=(0.08, 0.08), scale=(0.92, 1.08)),
        ])
    train_tfms.extend([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    eval_tfms = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    train_ds = RemappedImageFolder(DATA_ROOT, selected_class_ids, idx_to_new, transforms.Compose(train_tfms))
    val_ds = RemappedImageFolder(DATA_ROOT, selected_class_ids, idx_to_new, eval_tfms)
    test_ds = RemappedImageFolder(DATA_ROOT, selected_class_ids, idx_to_new, eval_tfms)
    return Subset(train_ds, train_idx), Subset(val_ds, val_idx), Subset(test_ds, test_idx)

def make_loaders(batch_size=32, augment=False):
    train_ds, val_ds, test_ds = make_datasets(augment=augment)
    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True),
        DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True),
        DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True),
    )

## 3. Wizualizacja danych

In [ ]:
class_counts = pd.Series(targets_new).value_counts().sort_index()
class_counts.index = selected_classes

plt.figure(figsize=(10, 4))
sns.barplot(x=class_counts.index, y=class_counts.values)
plt.xticks(rotation=35, ha="right")
plt.ylabel("Liczba obrazów")
plt.title("Rozkład wybranych klas")
plt.tight_layout()
plt.show()

In [ ]:
preview_ds = datasets.ImageFolder(DATA_ROOT, transform=transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
]))

plt.figure(figsize=(12, 6))
for plot_i, dataset_i in enumerate(selected_indices[:12], start=1):
    x, y_old = preview_ds[dataset_i]
    plt.subplot(3, 4, plot_i)
    plt.imshow(x.permute(1, 2, 0))
    plt.title(all_classes[y_old].split("___")[-1], fontsize=9)
    plt.axis("off")
plt.tight_layout()
plt.show()

## 4. Modele MLP

In [ ]:
def activation_layer(name):
    name = name.lower()
    if name == "relu":
        return nn.ReLU()
    if name == "tanh":
        return nn.Tanh()
    if name == "gelu":
        return nn.GELU()
    if name == "leaky_relu":
        return nn.LeakyReLU(0.1)
    raise ValueError(f"Nieznana aktywacja: {name}")

class MLP(nn.Module):
    def __init__(
        self,
        input_shape=(3, IMG_SIZE, IMG_SIZE),
        num_classes=NUM_CLASSES,
        hidden1=512,
        hidden2=None,
        activation="relu",
        dropout=0.0,
        use_batchnorm=False,
    ):
        super().__init__()
        in_features = int(np.prod(input_shape))
        layers = [nn.Flatten(), nn.Linear(in_features, hidden1)]
        if use_batchnorm:
            layers.append(nn.BatchNorm1d(hidden1))
        layers.append(activation_layer(activation))
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        if hidden2 is not None:
            layers.append(nn.Linear(hidden1, hidden2))
            if use_batchnorm:
                layers.append(nn.BatchNorm1d(hidden2))
            layers.append(activation_layer(activation))
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            layers.append(nn.Linear(hidden2, num_classes))
        else:
            layers.append(nn.Linear(hidden1, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

## 5. Pętla treningowa i metryki

In [ ]:
def build_optimizer(model, name="adam", lr=1e-3, weight_decay=0.0):
    name = name.lower()
    if name == "sgd":
        return torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
    if name == "adam":
        return torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    if name == "adamw":
        return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    if name == "rmsprop":
        return torch.optim.RMSprop(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
    raise ValueError(f"Nieznany optimizer: {name}")

def run_epoch(model, loader, criterion, optimizer=None):
    train_mode = optimizer is not None
    model.train(train_mode)
    total_loss, total_correct, total = 0.0, 0, 0

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        if train_mode:
            optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        if train_mode:
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * x.size(0)
        total_correct += (logits.argmax(dim=1) == y).sum().item()
        total += x.size(0)

    return total_loss / total, total_correct / total

def train_model(config, epochs=10, verbose=True):
    train_loader, val_loader, test_loader = make_loaders(
        batch_size=config["batch_size"],
        augment=config.get("augment", False),
    )
    model = MLP(
        hidden1=config.get("hidden1", 512),
        hidden2=config.get("hidden2"),
        activation=config.get("activation", "relu"),
        dropout=config.get("dropout", 0.0),
        use_batchnorm=config.get("batchnorm", False),
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss()
    optimizer = build_optimizer(
        model,
        name=config.get("optimizer", "adam"),
        lr=config.get("lr", 1e-3),
        weight_decay=config.get("weight_decay", 0.0),
    )

    history = []
    best_state = None
    best_val_acc = -1
    start = time.time()

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, criterion)
        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
        })
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = deepcopy(model.state_dict())
        if verbose:
            print(f"epoch {epoch:02d}: train_acc={train_acc:.3f}, val_acc={val_acc:.3f}")

    model.load_state_dict(best_state)
    test_loss, test_acc = run_epoch(model, test_loader, criterion)
    result = {
        **config,
        "best_val_acc": best_val_acc,
        "test_acc": test_acc,
        "test_loss": test_loss,
        "time_sec": time.time() - start,
    }
    return model, pd.DataFrame(history), result

## 6. Eksperyment bazowy i wizualizacje dokładnościowe

In [ ]:
baseline_config = {
    "name": "baseline_mlp_1_hidden",
    "batch_size": 32,
    "optimizer": "adam",
    "lr": 1e-3,
    "hidden1": 512,
    "hidden2": None,
    "activation": "relu",
    "dropout": 0.0,
    "weight_decay": 0.0,
    "batchnorm": False,
    "augment": False,
}

baseline_model, baseline_history, baseline_result = train_model(baseline_config, epochs=10)
baseline_result

In [ ]:
def plot_history(history, title="Przebieg uczenia"):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history["epoch"], history["train_loss"], label="train")
    axes[0].plot(history["epoch"], history["val_loss"], label="val")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoka")
    axes[0].legend()

    axes[1].plot(history["epoch"], history["train_acc"], label="train")
    axes[1].plot(history["epoch"], history["val_acc"], label="val")
    axes[1].set_title("Accuracy")
    axes[1].set_xlabel("Epoka")
    axes[1].legend()
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

plot_history(baseline_history, "MLP baseline")

In [ ]:
def predict_all(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for x, y in loader:
            logits = model(x.to(DEVICE))
            y_pred.extend(logits.argmax(dim=1).cpu().numpy())
            y_true.extend(y.numpy())
    return np.array(y_true), np.array(y_pred)

_, _, test_loader = make_loaders(batch_size=64, augment=False)
y_true, y_pred = predict_all(baseline_model, test_loader)

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=selected_classes, yticklabels=selected_classes)
plt.xticks(rotation=35, ha="right")
plt.yticks(rotation=0)
plt.xlabel("Predykcja")
plt.ylabel("Klasa prawdziwa")
plt.title("Confusion matrix: MLP baseline")
plt.tight_layout()
plt.show()

print(classification_report(y_true, y_pred, target_names=selected_classes))

## 7. Siatka eksperymentów hiperparametrów

In [ ]:
experiments = [
    # Batch size
    {**baseline_config, "name": "batch_16", "batch_size": 16},
    {**baseline_config, "name": "batch_64", "batch_size": 64},

    # Optimizer i krok uczenia
    {**baseline_config, "name": "sgd_lr_1e-2", "optimizer": "sgd", "lr": 1e-2},
    {**baseline_config, "name": "adam_lr_1e-4", "optimizer": "adam", "lr": 1e-4},
    {**baseline_config, "name": "adamw_lr_1e-3", "optimizer": "adamw", "lr": 1e-3},
    {**baseline_config, "name": "rmsprop_lr_1e-3", "optimizer": "rmsprop", "lr": 1e-3},

    # Regularyzacja
    {**baseline_config, "name": "dropout_03", "dropout": 0.3},
    {**baseline_config, "name": "weight_decay_1e-4", "weight_decay": 1e-4},
    {**baseline_config, "name": "batchnorm", "batchnorm": True},
    {**baseline_config, "name": "dropout_wd_aug", "dropout": 0.3, "weight_decay": 1e-4, "augment": True},

    # Druga warstwa ukryta
    {**baseline_config, "name": "two_hidden_512_256", "hidden2": 256},
    {**baseline_config, "name": "two_hidden_regularized_aug", "hidden2": 256, "dropout": 0.3, "weight_decay": 1e-4, "augment": True},

    # Aktywacje
    {**baseline_config, "name": "tanh", "activation": "tanh"},
    {**baseline_config, "name": "gelu", "activation": "gelu"},
    {**baseline_config, "name": "leaky_relu", "activation": "leaky_relu"},
]

all_results = [baseline_result]
histories = {"baseline_mlp_1_hidden": baseline_history}
models = {"baseline_mlp_1_hidden": baseline_model}

EPOCHS_PER_EXPERIMENT = 8

for cfg in experiments:
    print("\n==", cfg["name"], "==")
    model, history, result = train_model(cfg, epochs=EPOCHS_PER_EXPERIMENT, verbose=False)
    print(f"best_val_acc={result['best_val_acc']:.3f}, test_acc={result['test_acc']:.3f}")
    all_results.append(result)
    histories[cfg["name"]] = history
    models[cfg["name"]] = model

results_df = pd.DataFrame(all_results).sort_values("test_acc", ascending=False)
results_df

In [ ]:
cols = [
    "name", "batch_size", "optimizer", "lr", "hidden1", "hidden2",
    "activation", "dropout", "weight_decay", "batchnorm", "augment",
    "best_val_acc", "test_acc", "time_sec",
]
final_table = results_df[cols].copy()
final_table["best_val_acc"] = final_table["best_val_acc"].round(4)
final_table["test_acc"] = final_table["test_acc"].round(4)
final_table["time_sec"] = final_table["time_sec"].round(1)
final_table

In [ ]:
plt.figure(figsize=(12, 5))
sns.barplot(data=final_table, x="test_acc", y="name")
plt.xlabel("Dokładność na zbiorze testowym")
plt.ylabel("Eksperyment")
plt.title("Porównanie wariantów MLP")
plt.xlim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
best_name = final_table.iloc[0]["name"]
plot_history(histories[best_name], f"Najlepszy wariant: {best_name}")

_, _, test_loader = make_loaders(batch_size=64, augment=False)
y_true, y_pred = predict_all(models[best_name], test_loader)
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", xticklabels=selected_classes, yticklabels=selected_classes)
plt.xticks(rotation=35, ha="right")
plt.yticks(rotation=0)
plt.xlabel("Predykcja")
plt.ylabel("Klasa prawdziwa")
plt.title(f"Confusion matrix: {best_name}")
plt.tight_layout()
plt.show()

## 8. Wizualizacja augmentacji

Poniżej pokazujemy kilka losowych transformacji dla tego samego obrazu. W sprawozdaniu warto zestawić wynik wariantu bez augmentacji i z augmentacją.

In [ ]:
raw_image_path, raw_label = full_dataset.samples[selected_indices[0]]
raw_img = Image.open(raw_image_path).convert("RGB")

aug_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10),
    transforms.RandomAffine(degrees=0, translate=(0.08, 0.08), scale=(0.92, 1.08)),
])

plt.figure(figsize=(10, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(aug_tfm(raw_img))
    plt.axis("off")
plt.suptitle("Przykładowe augmentacje")
plt.tight_layout()
plt.show()

## 9. Dodatkowy wariant CNN i wizualizacja filtrów splotowych

MLP po spłaszczeniu obrazu nie uczy filtrów splotowych. Aby spełnić wymaganie wizualizacji filtrów, trenujemy małą sieć CNN i pokazujemy filtry pierwszej warstwy `Conv2d`.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * (IMG_SIZE // 4) * (IMG_SIZE // 4), 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

def train_cnn(epochs=8, batch_size=32, lr=1e-3, augment=True):
    train_loader, val_loader, test_loader = make_loaders(batch_size=batch_size, augment=augment)
    model = SmallCNN().to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    history = []
    best_state, best_val_acc = None, -1

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, criterion)
        history.append({
            "epoch": epoch, "train_loss": train_loss, "train_acc": train_acc,
            "val_loss": val_loss, "val_acc": val_acc,
        })
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = deepcopy(model.state_dict())
        print(f"epoch {epoch:02d}: train_acc={train_acc:.3f}, val_acc={val_acc:.3f}")

    model.load_state_dict(best_state)
    test_loss, test_acc = run_epoch(model, test_loader, criterion)
    print("CNN test_acc:", round(test_acc, 4))
    return model, pd.DataFrame(history), test_acc

cnn_model, cnn_history, cnn_test_acc = train_cnn()
plot_history(cnn_history, "Small CNN")

In [ ]:
def visualize_conv_filters(model):
    first_conv = None
    for module in model.modules():
        if isinstance(module, nn.Conv2d):
            first_conv = module
            break
    if first_conv is None:
        raise ValueError("Model nie ma warstwy Conv2d.")

    weights = first_conv.weight.detach().cpu()
    weights = (weights - weights.min()) / (weights.max() - weights.min() + 1e-8)
    n = min(weights.shape[0], 16)

    plt.figure(figsize=(8, 8))
    for i in range(n):
        filt = weights[i].permute(1, 2, 0).numpy()
        plt.subplot(4, 4, i + 1)
        plt.imshow(filt)
        plt.title(f"filter {i}")
        plt.axis("off")
    plt.suptitle("Filtry pierwszej warstwy splotowej")
    plt.tight_layout()
    plt.show()

visualize_conv_filters(cnn_model)

## 10. Eksport tabeli wyników

Tabela końcowa zawiera konfiguracje hiperparametrów i dokładność. Plik CSV można dołączyć do sprawozdania.

In [ ]:
final_table.to_csv("plantvillage_mlp_results.csv", index=False)
final_table

## Wnioski do opisania w sprawozdaniu

Po uruchomieniu notebooka warto krótko porównać:

- wpływ `batch_size` na stabilność i czas uczenia,
- różnice między `SGD`, `Adam`, `AdamW` i `RMSprop`,
- wpływ `lr`, szczególnie zbyt małego lub zbyt dużego kroku,
- regularyzację przez `dropout`, `weight_decay` i `batchnorm`,
- skuteczność augmentacji względem wariantu bez augmentacji,
- wariant z drugą warstwą ukrytą względem bazowego MLP,
- aktywacje `relu`, `tanh`, `gelu`, `leaky_relu`,
- fakt, że filtry splotowe dotyczą modelu CNN, a nie MLP.